# Apply the Llama model 70B online

## Information

Inference API of Hugging Face exposes models that have large community interest and are in active use: https://huggingface.co/docs/api-inference/supported-models


**Precondition**: create an Access Token (https://huggingface.co/settings/tokens), set up a pro account to use the larger LLMs like Llama-3-70B (https://huggingface.co/pricing#pro) and accept the META LLAMA 3 COMMUNITY LICENSE AGREEMENT for the two different Llama models:

* for `meta-llama/Meta-Llama-3-70B-Instruct`: https://huggingface.co/meta-llama/Meta-Llama-3-70B-Instruct

Remark: Llama models are published under the **META LLAMA 3 COMMUNITY LICENSE AGREEMENT**. The Meta Llama 3 Community License grants users a non-exclusive, royalty-free license (you not need to pay ongoing fees) to use, modify, and distribute Llama 3 materials, with requirements for attribution and naming conventions when creating derivative works. Users with over 700 million monthly active users need a separate license, and Meta disclaims all warranties and limits liability for any use of the materials.


***
**Coding sources**


* You can run the `meta-llama/Meta-Llama-3-70B-Instruct`, see model page: https://huggingface.co/meta-llama/Meta-Llama-3-70B-Instruct
    + Hugging Face documentation: https://huggingface.co/docs/transformers/main/en/model_doc/llama3

## Load necessary libraries and data:

In [1]:
import os
import sys

# Assuming 'src' is two levels down (in the current directory or a subdirectory)
path_to_src = os.path.join('..')  # Moves one level down to 'src' folder

# Change the working directory
os.chdir(path_to_src)

# Verify the updated working directory
print("Updated working directory:", os.getcwd())

# load helper functions
import src.API_key as keys

import src.prompt_functions as pf
import src.prompt_text as pt
import src.check_data as check_data

Updated working directory: /home/fenn/Desktop/Publications/basal attributes/Analyses/part_III


load external data:

words of the single partitions and their respective mean_valence:

In [2]:
import pandas as pd

df_partitions = pd.read_excel('./data/LeidenAlgorithm_solution.xlsx')
df_partitions.rename(columns={'parition': 'partition'}, inplace=True)

In [3]:
df_partitions

,partition,words,mean_valence,sd_valence
0,1,aktive Formänderung durch Umwelteinwirkung,0.18,1.140892
1,1,reaktionsfähig,0.67,1.111209
2,1,passive Formänderung durch Umwelteinwirkung,-0.02,1.131126
3,1,autonom,0.75,0.949819
4,1,passive Verhaltensänderung durch Umwelteinwirkung,0.06,1.056184
5,1,aktive Verhaltensänderung durch Umwelteinwirkung,0.27,1.148185
6,1,intelligent,1.35,1.103033
7,1,multifunktional,1.65,1.024702
8,1,technologisch,0.76,0.959081
9,2,zuverlässig,2.17,1.017770


In [4]:
import pandas as pd

# Assuming df_partitions is already defined
group_stats = df_partitions.groupby('partition').agg(
    group_mean_valence=('mean_valence', 'mean'),
    group_sd_valence=('mean_valence', 'std')  # Standard deviation of mean_valence within each partition
).reset_index()

print(group_stats)

   partition  group_mean_valence  group_sd_valence
0          1             0.63000          0.577841
1          2             1.88375          0.208117
2          3            -1.87000          0.646942
3          4             1.80750          0.934144
4          5             1.68250          0.292390
5          6             0.33000          0.450000


combination of partitions I want to generate a text from:

In [ ]:
df_hypothesis= pd.read_excel('./data/LeidenAlgorithm_hypotheses.xlsx')
df_hypothesis.rename(columns={'partitionA': 'partitionA', 'partitionB': 'partitionB'}, inplace=True)


In [ ]:
df_hypothesis = df_hypothesis[df_hypothesis['choose'] == 1][['partitionA', 'partitionB']]
df_hypothesis = df_hypothesis.reset_index(drop=True)
df_hypothesis

,partitionA,partitionB
0,3,6
1,3,5
2,1,6
3,1,5
4,1,2
5,2,4


In [ ]:
import pandas as pd

# Assuming df_partitions and df_hypothesis are already defined

# Function to compute mean and std for combined partitions
def compute_partition_stats(row):
    partition_a = row['partitionA']
    partition_b = row['partitionB']
    
    # Filter df_partitions for the two partitions
    subset = df_partitions[df_partitions['partition'].isin([partition_a, partition_b])]
    
    # Compute mean and std of mean_valence
    mean_val = subset['mean_valence'].mean()
    std_val = subset['mean_valence'].std()
    
    return pd.Series({'computed_mean': mean_val, 'computed_sd': std_val})

# Apply function to each row in df_hypothesis
df_hypothesis[['computed_mean', 'computed_sd']] = df_hypothesis.apply(compute_partition_stats, axis=1)

df_hypothesis


,partitionA,partitionB,computed_mean,computed_sd
0,3,6,-0.927143,1.288264
1,3,5,-0.093750,1.954942
2,1,6,0.555000,0.545952
3,1,5,0.953846,0.706830
4,1,2,1.220000,0.775879
5,2,4,1.858333,0.516682


## Load the prompt template and model:

prompt template:

In [ ]:
from importlib import reload

pt = reload(pt)
print("pt.user_template:\n", pt.user_template)
print("\npt.system_template:\n", pt.system_template)

pt.user_template:
 
<Liste der zu verwendenden basalen Attribute: 
({items_list})>


pt.system_template:
 
<Aufgabe:
Die R&D-Abteilung hat mehrere Versionen des Jackensystems Nano-Pat-Parka entwickelt. In diesem frühen Entwicklungsstadium sollen Probanden erstes Feedback zu den Konzepten geben, 
die durch unterschiedliche Kombinationen basaler Attribute charakterisiert sind.
>

<Definition basaler Attribute:
Basale Attribute sind adjektivische Eigenschaften, mit denen grundlegende semantische und emotionale Merkmale neuer Technologien beschrieben werden.
>

<Struktur (One-Shot):
1. Einleitungssatz (ein Satz): Beschreibe den Anwendungsbereich der Schutzkleidung und integriere ein erstes Attribut aus der übergebenen Liste.
2. Hauptteil (zwei Sätze): Kombiniere jeweils zwei Attribute aus der Liste, so dass alle genau einmal verwendet werden.
3. Anwendungsszenario (ein Satz): Nenne kurz eine konkrete Anwendung, z. B. “Bei starkem Wind …”.
4. Abschlusssatz (ein Satz): Fasse die wesentliche 

model:

In [ ]:
pf = reload(pf)
pf.huggingface_API_call

<function src.prompt_functions.huggingface_API_call(prompt, items_list, api_key, model_name='meta-llama/llama-3.3-70b-instruct', json_schema=None, max_tokens=1000, temperature=0.0, verbose=True)>

## Run the model:

a non reasoning model is currently not working:

In [ ]:
row = df_hypothesis.iloc[5]

# Ensure the value is list-like
partitionA = [row['partitionA']]
partitionB = [row['partitionB']]
combined_partitions = list(partitionA) + list(partitionB)

combined_subset = df_partitions[df_partitions['partition'].isin(combined_partitions)]["words"].values
combined_subset = " // ".join(combined_subset.tolist())
combined_subset

'zuverlässig // wartungsfrei // selbstheilend // widerstandsfähig // selbstreparierend // haltbar // robust // langlebig // ökologisch // elektronikfrei // nachhaltig // umweltfreundlich'

a non reasoning model is currently not working:

In [ ]:
result = pf.huggingface_API_call(prompt=pt.prompt_template,
                     items_list=combined_subset,
                     api_key=keys.hugging_api_key,
                     model_name="meta-llama/llama-3.3-70b-instruct",  max_tokens=3000, temperature=0)

Tokens Used: 731
	Prompt Tokens: 545
	Completion Tokens: 186
Successful Requests: 1
Total Cost (USD): $0.0
Total Tokens: 731
Prompt Tokens: 545
Completion Tokens: 186
Total Cost (USD): $0.0


In [ ]:
print(result.content)

word_count = len(result.content.split())
print("Number of words:", word_count)

Die Schutzkleidung ist für extreme Bedingungen konzipiert und zeichnet sich durch ihre zuverlässige Funktionalität aus. 
Sie kombiniert widerstandsfähige und selbstheilende Eigenschaften mit robusten und langlebigen Materialien, sowie wartungsfrei und selbstreparierend mit haltbar und ökologisch. 
Bei starkem Wind schützt sie den Träger effektiv. 
Er ermöglicht somit nachhaltige und umweltfreundliche Lösungen, die auch elektronikfrei sind. 

**Attributs-Check:** zuverlässig, wartungsfrei, selbstheilend, widerstandsfähig, selbstreparierend, haltbar, robust, langlebig, ökologisch, elektronikfrei, nachhaltig, umweltfreundlich
Number of words: 66


a reasoning model is working:

In [ ]:
result = pf.huggingface_API_call(prompt=pt.prompt_template,
                     items_list=combined_subset,
                     api_key=keys.hugging_api_key,
                     model_name="deepseek/deepseek-r1-turbo",  max_tokens=3000, temperature=0)

Tokens Used: 3379
	Prompt Tokens: 497
	Completion Tokens: 2882
Successful Requests: 1
Total Cost (USD): $0.0
Total Tokens: 3379
Prompt Tokens: 497
Completion Tokens: 2882
Total Cost (USD): $0.0


In [ ]:
import re

def parse_result_content(content):
    def extract_between_tags(tag, text):
        match = re.search(fr"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
        return match.group(1).strip() if match else None

    def extract_after_think(text):
        pattern = (
            r"</think>\s*\n+"  # End of <think> block
            r"(.*?)"           # Capture content after <think>
            r"(?=\n+\*{1,2}(Verwendete Attribute:|Wortanzahl:)|\n+#)"  # Stop before known markers
        )
        match = re.search(pattern, text, re.DOTALL)
        return match.group(1).strip() if match else None

    def extract_word_count(text):
        match = re.search(r"\*Wortanzahl:\s*(\d+)\*", text)
        return int(match.group(1)) if match else None

    def check_attributes_confirmed(text):
        return bool(re.search(r"\*Alle vorgegebenen Attribute integriert.*\*", text))

    think_block = extract_between_tags("think", content)
    german_description = extract_after_think(content)
    word_count_declared = extract_word_count(content)
    attributes_confirmed = check_attributes_confirmed(content)
    word_count_actual = len(german_description.split()) if german_description else None

    return {
        "think_block": think_block,
        "german_description": german_description,
        "word_count_declared": word_count_declared,
        "word_count_actual": word_count_actual,
        "attributes_confirmed": attributes_confirmed
    }


# Example usage:
parsed_data = parse_result_content(result.content)

# Print extracted info
print("Think block:\n", parsed_data["think_block"])
print("\nGerman description:\n", parsed_data["german_description"])
print("\nNumber of words by LLM (declared):", parsed_data["word_count_declared"])
print("Number of words counted (actual):", parsed_data["word_count_actual"])
print("All attributes confirmed?:", parsed_data["attributes_confirmed"])

Think block:
 Okay, ich muss die Aufgabe Schritt für Schritt durchgehen. Zuerst prüfe ich die Anforderungen. Die R&D-Abteilung möchte Feedback zu verschiedenen Versionen des Nano-Pat-Parka, beschrieben durch basale Attribute. Die Struktur ist vorgegeben: Einleitungssatz, Hauptteil mit zwei Sätzen, Anwendungsszenario und Abschlusssatz. Alle Attribute müssen genau einmal verwendet werden, und die Valenz muss einheitlich sein.

Zuerst liste ich die gegebenen Attribute auf: zuverlässig, wartungsfrei, selbstheilend, widerstandsfähig, selbstreparierend, haltbar, robust, langlebig, ökologisch, elektronikfrei, nachhaltig, umweltfreundlich. Das sind zwölf Attribute. Ich muss sicherstellen, dass jedes genau einmal im Text vorkommt.

Die Einleitung soll den Anwendungsbereich nennen und ein erstes Attribut integrieren. Vielleicht "zuverlässig" als Einstieg, da es Vertrauen vermittelt. 

Im Hauptteil müssen jeweils zwei Attribute pro Satz kombiniert werden, insgesamt vier Attribute. Hier könnte ich

In [ ]:
print(result.content)

<think>
Okay, ich muss die Aufgabe Schritt für Schritt durchgehen. Zuerst prüfe ich die Anforderungen. Die R&D-Abteilung möchte Feedback zu verschiedenen Versionen des Nano-Pat-Parka, beschrieben durch basale Attribute. Die Struktur ist vorgegeben: Einleitungssatz, Hauptteil mit zwei Sätzen, Anwendungsszenario und Abschlusssatz. Alle Attribute müssen genau einmal verwendet werden, und die Valenz muss einheitlich sein.

Zuerst liste ich die gegebenen Attribute auf: zuverlässig, wartungsfrei, selbstheilend, widerstandsfähig, selbstreparierend, haltbar, robust, langlebig, ökologisch, elektronikfrei, nachhaltig, umweltfreundlich. Das sind zwölf Attribute. Ich muss sicherstellen, dass jedes genau einmal im Text vorkommt.

Die Einleitung soll den Anwendungsbereich nennen und ein erstes Attribut integrieren. Vielleicht "zuverlässig" als Einstieg, da es Vertrauen vermittelt. 

Im Hauptteil müssen jeweils zwei Attribute pro Satz kombiniert werden, insgesamt vier Attribute. Hier könnte ich "wart

In [ ]:
parsed_data["german_description"]

### check for missing words:

In [ ]:
check_data = reload(check_data)

In [ ]:
combined_subset

'zuverlässig // wartungsfrei // selbstheilend // widerstandsfähig // selbstreparierend // haltbar // robust // langlebig // ökologisch // elektronikfrei // nachhaltig // umweltfreundlich'

In [ ]:
parsed_data["german_description"]

In [ ]:
description = "Der Nano-Pat-Parka bietet **zuverlässigen** Schutz für Outdoor-Profis in extremen Klimazonen. Seine **wartungsfreie**, **selbstheilende** Oberfläche vereint **widerstandsfähige** Materialien mit **selbstreparierenden** Mikrostrukturen. Die **robuste**, **langlebige** Konstruktion aus **ökologischen** Fasern ist **elektronikfrei** und überzeugt durch **nachhaltige** Produktion sowie **umweltfreundliche** Entsorgung. Bei starkem Wind bewährt sich die **haltbare** Struktur durch stabilen Halt. Die Innovation liegt in der symbiotischen Verbindung natürlicher Ressourcen mit fortschrittlicher Materialwissenschaft.  "

In [ ]:
result_text = check_data.check_for_missing_matching_words(description, combined_subset, max_distance=3)
print(result_text)

Missing Words:
None

Partial Matches:
- 'zuverlässig' ~ 'zuverlässigen' (Levenshtein distance = 2)
- 'wartungsfrei' ~ 'wartungsfreie' (Levenshtein distance = 1)
- 'selbstheilend' ~ 'selbstheilende' (Levenshtein distance = 1)
- 'widerstandsfähig' ~ 'widerstandsfähige' (Levenshtein distance = 1)
- 'selbstreparierend' ~ 'selbstreparierenden' (Levenshtein distance = 2)
- 'haltbar' ~ 'haltbare' (Levenshtein distance = 1)
- 'robust' ~ 'robuste' (Levenshtein distance = 1)
- 'langlebig' ~ 'langlebige' (Levenshtein distance = 1)
- 'ökologisch' ~ 'ökologischen' (Levenshtein distance = 2)
- 'nachhaltig' ~ 'nachhaltige' (Levenshtein distance = 1)
- 'umweltfreundlich' ~ 'umweltfreundliche' (Levenshtein distance = 1)



In [ ]:
ERROR

### loop through the partitions and generate a text for each partition:

In [ ]:
for index, row in df_hypothesis.iterrows():
    print(f"Index {index}")

Index 0
Index 1
Index 2
Index 3
Index 4
Index 5


In [ ]:
# Initialize empty list to store all results
results_list = []


for index, row in df_hypothesis.iterrows():
    print(f"Index {index}")

    # Skip the first X iterations
    #if index < 4:
    #    continue

    # Ensure the value is list-like
    partitionA = [row['partitionA']]
    partitionB = [row['partitionB']]
    combined_partitions = list(partitionA) + list(partitionB)

    combined_subset = df_partitions[df_partitions['partition'].isin(combined_partitions)]["words"].values
    combined_subset = " // ".join(combined_subset.tolist())
    
    
    # Call the API
    result = pf.huggingface_API_call(
        prompt=pt.prompt_template,
        items_list=combined_subset,
        api_key=keys.hugging_api_key,
        model_name="deepseek/deepseek-r1-turbo",
        max_tokens=3500,
        temperature=0
    )

    # Parse the content
    parsed_data = parse_result_content(result.content)

    # Calculate actual word count manually
    if parsed_data["german_description"]:
        actual_word_count = len(parsed_data["german_description"].split())
    else:
        actual_word_count = None

    # Store everything
    results_list.append({
        "index": index,
        "partitionA": row['partitionA'],
        "partitionB": row['partitionB'],
        "combined_subset": combined_subset,
        "raw_content": result.content,
        "think_block": parsed_data["think_block"],
        "german_description": parsed_data["german_description"],
        "word_count_reported": parsed_data["word_count_declared"],
        "word_count_actual": parsed_data["word_count_actual"],
        "attributes_confirmed": parsed_data["attributes_confirmed"]
    })

Index 0
Index 1
Index 2
Index 3
Index 4
Tokens Used: 3532
	Prompt Tokens: 546
	Completion Tokens: 2986
Successful Requests: 1
Total Cost (USD): $0.0
Total Tokens: 3532
Prompt Tokens: 546
Completion Tokens: 2986
Total Cost (USD): $0.0
Index 5
Tokens Used: 3920
	Prompt Tokens: 497
	Completion Tokens: 3423
Successful Requests: 1
Total Cost (USD): $0.0
Total Tokens: 3920
Prompt Tokens: 497
Completion Tokens: 3423
Total Cost (USD): $0.0


In [ ]:
# results_list_backup = results_list

In [ ]:
results_list

[{'index': 0,
  'partitionA': 3.0,
  'partitionB': 6.0,
  'combined_subset': 'wartungsintensiv // enthält Kunststoff // leicht zerstörbar // umweltschädlich // Insekten ähnlich // bioinspiriert // lebensähnlich',
  'raw_content': '<think>\nOkay, let\'s tackle this. The user wants a product description for the Nano-Pat-Parka using all the given attributes. First, I need to check each attribute: wartungsintensiv, enthält Kunststoff, leicht zerstörbar, umweltschädlich, Insekten ähnlich, bioinspiriert, lebensähnlich. Wait, some of these are negative like umweltschädlich and leicht zerstörbar. But the instructions say all attributes must have the same emotional valence. That\'s a problem. How can I present negative attributes in a positive light? Maybe the product has these features but the description frames them as necessary trade-offs or highlights other positive aspects.\n\nNext, structure: one intro sentence with an attribute, two main sentences each combining two attributes, an applic

In [ ]:
for row in results_list:
    print(f"Index {row['index']}:")
    print(f"{row['raw_content']}")

In [ ]:
# Create a DataFrame from all collected results
df_results = pd.DataFrame(results_list)

# Save as CSV
df_results.to_csv("output/LLM_results.csv", index=False, encoding="utf-8-sig")

# Save as Excel
df_results.to_excel("output/LLM_results.xlsx", index=False)

print("Results saved successfully!")

Results saved successfully!


## check the generated text for missing words:

In [ ]:
import pandas as pd


# read the first sheet into a DataFrame
df = pd.read_excel("output/LLM_results_manually.xlsx", sheet_name=0)

# inspect
print(df.head())

   index  partitionA  partitionB  \
0      0           3           6   
1      1           3           5   
2      2           1           6   
3      3           1           5   
4      4           1           2   

                                     combined_subset  \
0  wartungsintensiv // enthält Kunststoff // leic...   
1  wartungsintensiv // enthält Kunststoff // leic...   
2  aktive Formänderung durch Umwelteinwirkung // ...   
3  aktive Formänderung durch Umwelteinwirkung // ...   
4  aktive Formänderung durch Umwelteinwirkung // ...   

                                         raw_content  \
0  <think>\nOkay, let's tackle this. The user wan...   
1  <think>\nOkay, let's tackle this. The user wan...   
2  <think>\nOkay, let's tackle this. The user wan...   
3  <think>\nOkay, let's tackle this. The user wan...   
4  <think>\nOkay, let's tackle this. The user wan...   

                                         think_block  german_description  \
0  Okay, let's tackle this. The u

In [ ]:
for index, row in df.iterrows():
    print(f"Index {index}")
    print("Combined subset:\n", row.combined_subset, "\n")
    result_text = check_data.check_for_missing_matching_words(row.final_text, row.combined_subset, max_distance=5)
    print(result_text)

Index 0
Combined subset:
 wartungsintensiv // enthält Kunststoff // leicht zerstörbar // umweltschädlich // Insekten ähnlich // bioinspiriert // lebensähnlich 

Exact Matches:
- 'enthalt kunststoff'
- 'insekten ahnlich'

Missing Words:
None

Partial Matches:
- 'wartungsintensiv' ~ 'wartungsintensiven' (Levenshtein distance = 2)
- 'leicht zerstorbar' ~ 'leicht zerstorbarer' (Levenshtein distance = 2)
- 'umweltschadlich' ~ 'umweltschadliche' (Levenshtein distance = 1)
- 'bioinspiriert' ~ 'bioinspirierte' (Levenshtein distance = 1)
- 'lebensahnlich' ~ 'lebensahnliche' (Levenshtein distance = 1)

Index 1
Combined subset:
 wartungsintensiv // enthält Kunststoff // leicht zerstörbar // umweltschädlich // Energie speichernd // energieeffizient // energieautonom // Energie generierend 

Exact Matches:
- 'wartungsintensiv'

Missing Words:
- 'umweltschadlich' was not found.

Partial Matches:
- 'enthalt kunststoff' ~ 'enthaltendem kunststoff' (Levenshtein distance = 5)
- 'leicht zerstorbar' ~ 'le